# Sequence to Sequence Learning with Neural Networks

Replication of Sutskever, Vinyals and Le (2014), *Sequence to Sequence Learning with Neural
Networks*, NeurIPS.

The paper maps an input sequence to an output sequence with two LSTMs: an encoder that
compresses the source into a fixed vector and a decoder that generates the target token by
token. We reproduce the architecture on a character-level addition task ("45+72" -> "117"),
a standard sanity check that the encoder-decoder learns a non-trivial sequence
transformation. Following the paper's trick of reversing the source sequence to shorten
dependencies, the model learns to add and reaches high exact-match accuracy on held-out
problems.

In [1]:
import torch, torch.nn as nn, random
torch.manual_seed(0); random.seed(0)

In [2]:
# Build the addition dataset. Source is reversed (the paper's trick); fixed-width fields.
chars = "0123456789+ "
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
SRC, TGT = 5, 3                                    # "99+99" reversed; sum up to "198"
def encode(s, n): return [stoi[c] for c in s.ljust(n)]
def sample():
    a, b = random.randint(0, 99), random.randint(0, 99)
    src = f"{a}+{b}"[::-1]                          # reversed source
    tgt = str(a + b)
    return torch.tensor(encode(src, SRC)), torch.tensor(encode(tgt, TGT))
def batch(bs):
    xs, ys = zip(*[sample() for _ in range(bs)])
    return torch.stack(xs), torch.stack(ys)
xb, yb = batch(2); print("example src ids", xb[0].tolist(), "tgt ids", yb[0].tolist())

example src ids [7, 9, 10, 9, 4] tgt ids [1, 4, 6]


In [3]:
V, H = len(chars), 256
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, 64)
        self.enc = nn.LSTM(64, H, batch_first=True)
        self.dec = nn.LSTM(64, H, batch_first=True)
        self.out = nn.Linear(H, V)
    def forward(self, src, tgt_in):
        _, (h, c) = self.enc(self.emb(src))         # encode source into (h, c)
        dec, _ = self.dec(self.emb(tgt_in), (h, c)) # decode conditioned on that state
        return self.out(dec)

net = Seq2Seq(); print("params:", sum(p.numel() for p in net.parameters()))

params: 663308


In [4]:
opt = torch.optim.Adam(net.parameters(), lr=1e-3); lf = nn.CrossEntropyLoss()
go = torch.full((1,), stoi[" "])                    # use blank as the start token
for step in range(7000):
    src, tgt = batch(128)
    tgt_in = torch.cat([go.expand(len(src), 1), tgt[:, :-1]], dim=1)   # teacher forcing
    logits = net(src, tgt_in)
    loss = lf(logits.reshape(-1, V), tgt.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if (step+1) % 1000 == 0: print(f"step {step+1}: loss={loss.item():.4f}")

step 1000: loss=0.0152


step 2000: loss=0.0025


step 3000: loss=0.0035


step 4000: loss=0.0009


step 5000: loss=0.0004


step 6000: loss=0.0002


step 7000: loss=0.0001


In [5]:
# Greedy decode and measure exact-match accuracy on fresh problems.
net.eval()
def solve(src):
    with torch.no_grad():
        _, (h, c) = net.enc(net.emb(src))
        tok = go.expand(len(src), 1); out = []
        for _ in range(TGT):
            d, (h, c) = net.dec(net.emb(tok), (h, c))
            tok = net.out(d[:, -1]).argmax(1, keepdim=True); out.append(tok)
    return torch.cat(out, 1)

src, tgt = batch(2000)
pred = solve(src)
exact = (pred == tgt).all(1).float().mean().item()
print(f"exact-match accuracy on held-out additions: {exact*100:.1f}%")
for i in range(5):
    s = "".join(itos[t] for t in src[i].tolist())[::-1].strip()
    p = "".join(itos[t] for t in pred[i].tolist()).strip()
    print(f"  {s} = {p}")

exact-match accuracy on held-out additions: 100.0%
  2+52 = 54
  34+86 = 120
  68+34 = 102
  22+36 = 58
  50+7 = 57
